In [21]:
from typing import Literal , TypedDict ,Annotated
from langgraph.graph import StateGraph , START, END
from langchain_ollama import ChatOllama
from pydantic import Field, BaseModel
from langgraph.graph.message import BaseMessage , add_messages
from langchain_core.messages import HumanMessage ,AIMessage
from langgraph.checkpoint.memory import MemorySaver
 
from langgraph.checkpoint.sqlite import SqliteSaver

import sqlite3
model = "qwen3.5:9b"
 

llm = ChatOllama(
    model=model,
    temperature=0
)
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]


def chat_node(state:ChatState)->ChatState:
    message = state['messages']
   
    response = llm.invoke(message)
    
    return{
        "messages":[response]
    }
    
connection = sqlite3.connect(database="chatbot.db",check_same_thread=False)    
checkpoint = SqliteSaver(connection)
graph = StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

# define edges
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
chatbot = graph.compile(checkpoint)
threads = checkpoint.list(None)
print(threads)


<generator object SqliteSaver.list at 0x000001EA4D8FB2A0>


In [22]:
def get_all_threads() -> list[str]:
    thread_ids = set()

    for checkpoint_tuple in checkpoint.list(None):
        thread_id = checkpoint_tuple.config["configurable"].get("thread_id")

        if thread_id:
            thread_ids.add(thread_id)

    return sorted(thread_ids)

In [7]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3.5:9b",
    temperature=0,
    reasoning=False,
    num_ctx=4096,
    num_predict=512,
    keep_alive="30m"
)

In [8]:
for chunk in llm.stream("what is pressure transmitter"):
    print(chunk.content,end="",flush=True)

A **pressure transmitter** is an industrial instrument used to measure fluid pressure and convert that physical measurement into a standardized electrical signal. This signal can then be transmitted to a control system, data logger, or display device for monitoring, recording, or process control.

Here is a detailed breakdown of how it works and its key components:

### 1. Core Function
The primary job of a pressure transmitter is to act as a bridge between the physical world (fluid pressure) and the digital/electrical world. It typically takes an input range (e.g., 0 to 10 bar) and outputs a linear electrical signal (commonly **4–20 mA**, **0–10 V**, or digital protocols like **HART** or **Foundation Fieldbus**).

### 2. How It Works
The operation generally follows these steps:
*   **Sensing:** A sensing element (usually a diaphragm) is exposed to the process pressure. When pressure is applied, the diaphragm deflects or deforms.
*   **Conversion:** This mechanical deformation changes 

In [11]:
for chunk in llm.stream("thank you"):
    print(chunk.content,end="",flush=True)

You're very welcome! 😊 Is there anything else I can help you with?

In [10]:
llm.invoke("thank you")

AIMessage(content="You're very welcome! 😊 Is there anything else I can help you with?", additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-08-26T11:40:51.7086926Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1501373200, 'load_duration': 1319000, 'prompt_eval_count': 14, 'prompt_eval_duration': 373658000, 'eval_count': 18, 'eval_duration': 1080644000, 'logprobs': None, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--01a03ddf-769e-7283-b241-9e206e7e4e49-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 18, 'total_tokens': 32})